# Práctica NoSQL: Python, Dask y MongoDB Atlas

**Equipo:** Alejandro Alvarenga, Eduardo Navarrete, Josué Melara · **Fecha:** 23-octubre-2026

## Requisitos de entorno
- **Python 3.10, 3.11 o 3.12** (recomendado 3.11 o 3.12). La celda de configuración falla con un mensaje claro si la versión es menor que 3.10.
- Un entorno virtual propio, por ejemplo `python -m venv .venv`, activado antes de abrir Jupyter.
- Cuenta de MongoDB Atlas (solo para la Parte 3). La URI no se escribe en este cuaderno: va en un archivo `.env` local que no se sube al repositorio.

## Reparto del trabajo
| Parte | Rol | Responsable | Entrega |
|---|---|---|---|
| 0 | Configuración común | Todos | Entorno funcionando |
| 1 | Ingeniero de Datos | [completar nombre] | Lógica Dask, limpieza, generación de datos, validación de filas |
| 2 | Arquitecto Cloud | [completar nombre] | Benchmark Pandas vs. Dask, entorno de ejecución, conectividad AWS/GCP |
| 3 | DBA NoSQL | Alejandro Alvarenga | Conexión Atlas, índice, upsert, 3 consultas |
| 4 | Registro y entrega | Todos | Aportes, commits, revisión cruzada, evidencia cloud |

## Revisión cruzada
| Parte revisada | Revisa |
|---|---|
| Parte 1 (Ingeniero de Datos) | DBA NoSQL |
| Parte 2 (Arquitecto Cloud) | Ingeniero de Datos |
| Parte 3 (DBA NoSQL) | Arquitecto Cloud |

Además, los tres ejecutan el cuaderno completo en su máquina.

**Orden de ejecución:** de arriba abajo. La Parte 3 necesita los archivos de `results/` que generan las Partes 1 y 2.

## Parte 0 · Preparación (todos)

### 0.1 Dependencias
Ejecuta esta celda **una vez** por entorno y luego **reinicia el kernel** (Kernel → Restart). Las versiones mínimas evitan incompatibilidades conocidas (cadenas de texto en Dask/Pandas, conexión `mongodb+srv://`, certificados TLS).

In [1]:
%pip install -q "dask[dataframe]>=2024.8" "pandas>=2.1" "numpy>=1.26" "pyarrow>=14" "pymongo>=4.6" "python-dotenv>=1.0" "certifi" "psutil"

Note: you may need to restart the kernel to use updated packages.


### 0.2 Configuración y funciones de apoyo
Aquí están todos los parámetros. Para una prueba rápida se puede bajar `RECORDS` (por ejemplo a `200_000`) y para un volumen mayor, subirlo. Si el volumen supera o no la RAM se determina con la medición de la Parte 2.

In [2]:
import sys, os, json, time, glob, platform, statistics
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import dask
import dask.dataframe as dd
import psutil

assert sys.version_info >= (3, 10), f"Se requiere Python 3.10 o superior; tienes {sys.version.split()[0]}"

# --- Parámetros (editables) ---
DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')
RECORDS     = 2_000_000     # filas totales a generar
FILES       = 16            # número de CSV
GEN_CHUNK   = 250_000       # filas por bloque al escribir cada CSV
CHUNKSIZE   = 250_000       # filas por bloque en el baseline secuencial de Pandas
BLOCKSIZE   = '64MB'        # tamaño de partición de Dask (un bloque no cruza archivos)
DIRTY_RATIO = 0.01          # fracción de filas con defectos inyectados a propósito
SEED        = 42
REPEATS     = 3             # repeticiones de cada medición
SCHEDULER   = 'threads'     # 'threads' | 'processes' | 'synchronous'
NUM_WORKERS = os.cpu_count()

REGIONS    = ['norte', 'sur', 'este', 'oeste', 'centro']
CATEGORIES = ['electronica', 'hogar', 'deportes', 'libros', 'ropa', 'alimentos']
GROUPS     = ['region', 'category', 'month']
USECOLS    = ['timestamp', 'region', 'category', 'amount', 'units']
DTYPES     = {'timestamp': 'string', 'region': 'string', 'category': 'string',
              'amount': 'float64', 'units': 'int64'}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def hardware_info():
    vm = psutil.virtual_memory()
    return {'sistema': platform.platform(), 'procesador': platform.processor() or platform.machine(),
            'nucleos_fisicos': psutil.cpu_count(logical=False), 'nucleos_logicos': psutil.cpu_count(logical=True),
            'ram_total_gb': round(vm.total / 1024**3, 2), 'ram_disponible_gb': round(vm.available / 1024**3, 2)}

print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| dask', dask.__version__, '| numpy', np.__version__)
print(hardware_info())

Python 3.15.1 | pandas 3.1.0 | dask 2026.8.0 | numpy 2.5.4
{'sistema': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'procesador': 'arm', 'nucleos_fisicos': 8, 'nucleos_logicos': 8, 'ram_total_gb': 8.0, 'ram_disponible_gb': 1.62}


### Aportes del equipo
| Miembro | Rol | Código aportado | Documentación aportada | Módulo que revisó | Commit(s) |
|---|---|---|---|---|---|
| [completar nombre] | Ingeniero de Datos | [completar] | [completar] | Parte 2 | [enlace] |
| [completar nombre] | Arquitecto Cloud | [completar] | [completar] | Parte 3 | [enlace] |
| Alejandro Alvarenga | DBA NoSQL | [completar] | [completar] | Parte 1 | [enlace] |

### Resultados medidos (copiar de la celda anterior)
- **Hardware y sistema:** [completar]
- **Filas y tamaño de entrada:** [completar]
- **Pandas / Dask / Speedup:** [completar] s / [completar] s / [completar] x
- **Equivalencia Pandas–Dask:** [completar]
- **Atlas:** ping [completar] · documentos cargados [completar] · salida de las 3 consultas [completar]

### Evidencia cloud AWS/GCP
- [ ] Recurso desplegado (tipo, región): [completar]
- [ ] Captura de la consola con el recurso en ejecución (fecha visible): [completar]
- [ ] Captura del trabajo ejecutándose en ese recurso: [completar]
- Si no se desplegó infraestructura cloud, se indica en el informe.
